# Data Quality
This notebook contains the code for data quality checks, including missing values, duplicates, types, inconsistent labels and EDA visualizations.

### Imports

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.data import load_raw, quality_report, label_issues, rows_to_drop, multiclass_label, preprocess, MODES

## General quality check

In [2]:
df = load_raw()
df.info()
quality_report(df).to_frame("count")

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

,count
rows,10000
missing_values,0
duplicate_rows,0
duplicate_measurements,0
duplicate_UDI,0
type_vs_product_id_mismatch,0
negative_values,0
process_temp_below_air,0


**Findings**
- 10000 rows, 14 columns, correct types.
- No missing values, as expected, no duplicate rows, no duplicate UDIs.
- `Type` always matches the first letter of `Product ID`. There are no negative values, and process temperature is never below air temperature.
> No imputation or cleaning of the features is needed.

## Label consistency

In [3]:
issues = label_issues(df)
display(issues["reason"].value_counts())
issues

reason
multiple_modes          23
mode_no_failure, rnf    18
failure_no_mode          9
multiple_modes, rnf      1
Name: count, dtype: int64

,UDI,Machine failure,TWF,HDF,PWF,OSF,RNF,reason
69,70,1,0,0,1,1,0,multiple_modes
1221,1222,0,0,0,0,0,1,"mode_no_failure, rnf"
1302,1303,0,0,0,0,0,1,"mode_no_failure, rnf"
1324,1325,1,0,0,1,1,0,multiple_modes
1437,1438,1,0,0,0,0,0,failure_no_mode
1496,1497,1,0,0,1,1,0,multiple_modes
1748,1749,0,0,0,0,0,1,"mode_no_failure, rnf"
2072,2073,0,0,0,0,0,1,"mode_no_failure, rnf"
2559,2560,0,0,0,0,0,1,"mode_no_failure, rnf"
2749,2750,1,0,0,0,0,0,failure_no_mode


**Findings**

`Machine failure` should be 1 if and only if at least one mode is active. That rule is broken in two ways:
- **9 rows `failure_no_mode`:** a failure with no mode. The failure label is valid, but there is no mode to assign.
- **18 rows `mode_no_failure`:** RNF active but `Machine failure = 0`. These are the only true inconsistencies.

Separately, **24 rows `multiple_modes`** have more than one mode at once. They are not errors, because the modes are independent and can happen together.

## Multiclass rows and final classes

In [4]:
multi_kept = df.drop(rows_to_drop(df, "multi"))
print(f"Multiclass: dropped {len(df) - len(multi_kept)} rows, kept {len(multi_kept)}")
multiclass_label(multi_kept).value_counts()

Multiclass: dropped 27 rows, kept 9973


No failure    9643
HDF            106
OSF             95
PWF             83
TWF             46
Name: count, dtype: int64

**Decisions**
- **Binary:** keep all 10000 rows and use `Machine failure` as given. The 18 `mode_no_failure` rows stay as no failure.
- **Multiclass (No failure, TWF, HDF, PWF, OSF):** drop 27 rows with no learnable label, the 9 `failure_no_mode` and the 18 RNF only. RNF is excluded because it is random.
- **Multi-mode rows:** instead of having both modes, the rarest one is the one given (TWF < OSF < PWF < HDF). This keeps 24 real failures and keeps these hard cases in the test set. The cost is that predicting the other true mode counts as an error, we will check this in the error analysis.

**Final classes**: 9643 No failure, 106 HDF, 95 OSF, 83 PWF, 46 TWF.

## Preprocessing Check

In [7]:
for task in ["binary", "multi"]:
    X, y = preprocess(df, task)
    print(task, X.shape, y.value_counts().sort_index().to_dict())

binary (10000, 9) {0: 9661, 1: 339}
multi (9973, 9) {0: 9643, 1: 46, 2: 106, 3: 83, 4: 95}


**Decisions**
- **Dropped columns:** `UDI` and `Product ID` are identifiers. The first letter of `Product ID` is already `Type`.
- **Engineered features**:
  - `Temp diff [K]` = process − air temperature -> HDF 
  - `Power [W]` = torque * rpm · 2π/60 -> PWF 
  - `Strain [min Nm]` = tool wear * torque -> OSF 
- **Labels:** binary uses `Machine failure`. Multiclass is integer-encoded in `CLASSES` order (0 = No failure, 1 = TWF, 2 = HDF, 3 = PWF, 4 = OSF)

**Result:** binary X is 10000 by 9 and multiclass X is 9973 by 9. Both become 11 columns after the preprocessor.